### RAG pipelines - Data ingestion to vector DB pipeline

In [14]:
import os

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13264\1427734644.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


### Read aill the PDF's inside the directory

In [15]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            # Add documents after processing all pages
            all_documents.extend(documents)

            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: 1512.03385v1.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/KGMNOB+TT93o00', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 45, '/FontDescriptor': IndirectObject(64, 0, 2340957878464), '/LastChar': 121, '/Widths': [333, 0, 0, 507, 0, 507, 0, 0, 507, 507, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 440, 0, 0, 0, 440, 0, 0, 0, 0, 0, 0, 280, 0, 0, 0, 0, 0, 333, 0, 0, 0, 0, 0, 0, 480]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/MKDMEG+TT55o00', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 45, '/FontDescriptor': IndirectObject(312, 0, 2340957878464), '/LastChar': 121, '/Widths': [340, 0, 0, 0, 500, 0, 500, 500, 0, 0, 0, 500, 0, 0, 0, 0, 

Loaded 12 pages

Processing: EJ1172284.pdf
Loaded 11 pages

Processing: IJRTI2304061.pdf
Loaded 5 pages

Processing: Research_Paper_On_Artificial_Intelligence_And_Its.pdf
Loaded 10 pages

Total documents loaded: 38


In [16]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.12', 'creator': 'LaTeX with hyperref package', 'creationdate': '2015-12-11T01:13:45+00:00', 'author': '', 'keywords': '', 'moddate': '2015-12-11T01:13:45+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.3-1.40.12 (TeX Live 2011) kpathsea version 6.0.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\1512.03385v1.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': '1512.03385v1.pdf', 'file_type': 'pdf'}, page_content='Deep Residual Learning for Image Recognition\nKaiming He Xiangyu Zhang Shaoqing Ren Jian Sun\nMicrosoft Research\n{kahe, v-xiangz, v-shren, jiansun}@microsoft.com\nAbstract\nDeeper neural networks are more difﬁcult to train. We\npresent a residual learning framework to ease the training\nof networks that are substantially deeper than those used\npreviously. We explicitly reformulate the layers as learn-\ning residual functions with reference to the layer inputs, in-\ns

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [18]:
chunks = split_documents(all_pdf_documents)
chunks

Split 38 documents into 196 chunks

Example chunk:
Content: Deep Residual Learning for Image Recognition
Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun
Microsoft Research
{kahe, v-xiangz, v-shren, jiansun}@microsoft.com
Abstract
Deeper neural networks are more...
Metadata: {'producer': 'pdfTeX-1.40.12', 'creator': 'LaTeX with hyperref package', 'creationdate': '2015-12-11T01:13:45+00:00', 'author': '', 'keywords': '', 'moddate': '2015-12-11T01:13:45+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.3-1.40.12 (TeX Live 2011) kpathsea version 6.0.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\1512.03385v1.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': '1512.03385v1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.12', 'creator': 'LaTeX with hyperref package', 'creationdate': '2015-12-11T01:13:45+00:00', 'author': '', 'keywords': '', 'moddate': '2015-12-11T01:13:45+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.3-1.40.12 (TeX Live 2011) kpathsea version 6.0.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\1512.03385v1.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': '1512.03385v1.pdf', 'file_type': 'pdf'}, page_content='Deep Residual Learning for Image Recognition\nKaiming He Xiangyu Zhang Shaoqing Ren Jian Sun\nMicrosoft Research\n{kahe, v-xiangz, v-shren, jiansun}@microsoft.com\nAbstract\nDeeper neural networks are more difﬁcult to train. We\npresent a residual learning framework to ease the training\nof networks that are substantially deeper than those used\npreviously. We explicitly reformulate the layers as learn-\ning residual functions with reference to the layer inputs, in-\ns

### embedding and vector DB store

In [2]:
import numpy as np
print("NumPy OK")

NumPy OK


In [3]:
from sentence_transformers import SentenceTransformer
print("Sentence Transformers OK")

Sentence Transformers OK


In [4]:
import chromadb
print("ChromaDB OK")

ChromaDB OK


In [5]:
from chromadb.config import Settings
print("Settings OK")

Settings OK


In [6]:
from sklearn.metrics.pairwise import cosine_similarity
print("Sklearn OK")

Sklearn OK


In [7]:
import numpy as np

from sentence_transformers import SentenceTransformer

import chromadb

from chromadb.config import Settings

import uuid

from typing import List, Dict, Any, Tuple

from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for SentenceTransformer
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)

            print(
                f"Model loaded successfully. "
                f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
            )

        except Exception as e:
            print(f"Error loading model: {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
         numpy array of embeddings with shape (len(texts), embedding_dim)
         """
        if not self.model:
            raise ValueError("Model not loaded.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    # def get_embedding_dimension(self) -> int:
    #     """Return the embedding dimension of the model"""
    #     if not self.model:
    #         raise ValueError("Model not loaded.")
    #     return self.model.get_sentence_embedding_dimension()

###initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager


     


Loading embedding model: all-MiniLM-L6-v2


c:\AIcourses\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\AIcourses\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_h

Model loaded successfully. Embedding dimension: 384


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13264\12100896.py:24: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"


#Vector store

In [12]:
import os
class VectorStore:
    """Manages document embeddings in a chromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
          #create persistent chromaD client
          os.makedirs(self.persist_directory, exist_ok=True)
          self.client = chromadb.PersistentClient(path=self.persist_directory)

          #Get or create the collection
          self.collection = self.client.get_or_create_collection(
              name=self.collection_name,
              metadata={"description": "PDF document embeddings for RAG"})
          print(f"Vector store initialized with collection: {self.collection_name}")
          print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):  
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of Langchain document
            embeddings: Corresponding numpy array of embeddings
        """
        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents and embeddings must match.")

        print(f"Adding {len(documents)} documents to the vector store...")

        #prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #generate UUID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata = dict(doc.metadata)  # Ensure it's a dict
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #prepare document text
            documents_text.append(doc.page_content)

            #prepare embeddings
            embeddings_list.append(embedding.tolist())

        #Add to ChromaDB collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            ) 
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore           

Vector store initialized with collection: pdf_documents
Existing documents in collection: 0


In [19]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.12', 'creator': 'LaTeX with hyperref package', 'creationdate': '2015-12-11T01:13:45+00:00', 'author': '', 'keywords': '', 'moddate': '2015-12-11T01:13:45+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.3-1.40.12 (TeX Live 2011) kpathsea version 6.0.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\1512.03385v1.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': '1512.03385v1.pdf', 'file_type': 'pdf'}, page_content='Deep Residual Learning for Image Recognition\nKaiming He Xiangyu Zhang Shaoqing Ren Jian Sun\nMicrosoft Research\n{kahe, v-xiangz, v-shren, jiansun}@microsoft.com\nAbstract\nDeeper neural networks are more difﬁcult to train. We\npresent a residual learning framework to ease the training\nof networks that are substantially deeper than those used\npreviously. We explicitly reformulate the layers as learn-\ning residual functions with reference to the layer inputs, in-\ns

In [21]:
### convert the text chunks into embeddings
texts = [doc.page_content for doc in chunks]
texts

['Deep Residual Learning for Image Recognition\nKaiming He Xiangyu Zhang Shaoqing Ren Jian Sun\nMicrosoft Research\n{kahe, v-xiangz, v-shren, jiansun}@microsoft.com\nAbstract\nDeeper neural networks are more difﬁcult to train. We\npresent a residual learning framework to ease the training\nof networks that are substantially deeper than those used\npreviously. We explicitly reformulate the layers as learn-\ning residual functions with reference to the layer inputs, in-\nstead of learning unreferenced functions. We provide com-\nprehensive empirical evidence showing that these residual\nnetworks are easier to optimize, and can gain accuracy from\nconsiderably increased depth. On the ImageNet dataset we\nevaluate residual nets with a depth of up to 152 layers—8×\ndeeper than VGG nets [41] but still having lower complex-\nity. An ensemble of these residual nets achieves 3.57% error\non the ImageNettest set. This result won the 1st place on the\nILSVRC 2015 classiﬁcation task. We also prese

In [22]:
### convert the text chunks into embeddings
texts = [doc.page_content for doc in chunks]
### Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)
###store the documents and embeddings in the vector store
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 196 texts...


Batches: 100%|██████████| 7/7 [00:38<00:00,  5.56s/it]


Generated embeddings with shape: (196, 384)
Adding 196 documents to the vector store...
Successfully added 196 documents to vector store
Total documents in collection: 196


In [24]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the RAGRetriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The input query string
            top_k: Number of top documents to retrieve
            score_threshold: Minimum similarity score for retrieved documents
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["metadatas", "documents", "distances"]
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity score
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "document": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} "
                    f"documents (after filtering)"
                )

            else:
                print("No documents retrieved.")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


# Create the RAG retriever
rag_retriever = RAGRetriever(
    vectorstore,
    embedding_manager
)

In [25]:
rag_retriever

In [26]:
rag_retriever.retrieve("What is Residual Representations?")

Retrieving documents for query: 'What is Residual Representations?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.06it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_5f9f814f_9',
  'document': 'layer residual net is the deepest network ever presented on\nImageNet, while still having lower complexity than VGG\nnets [41]. Our ensemble has 3.57% top-5 error on the\nImageNet test set, and won the 1st place in the ILSVRC\n2015 classiﬁcation competition . The extremely deep rep-\nresentations also have excellent generalization performance\non other recognition tasks, and lead us to further win the\n1st places on: ImageNet detection, ImageNet localization,\nCOCO detection, and COCO segmentation in ILSVRC &\nCOCO 2015 competitions. This strong evidence shows that\nthe residual learning principle is generic, and we expect that\nit is applicable in other vision and non-vision problems.\n2. Related Work\nResidual Representations. In image recognition, VLAD\n[18] is a representation that encodes by the residual vectors\nwith respect to a dictionary, and Fisher Vector [30] can be\nformulated as a probabilistic version [18] of VLAD. Both\nof them ar

In [27]:
rag_retriever.retrieve("What is Evolution of AI definition?")

Retrieving documents for query: 'What is Evolution of AI definition?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.01it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_3a4e1cdf_163',
  'document': 'features of artificial Intelligence, intr oduction, definitions of AI, history, applications, growth and \nachievements. \n \nEvolution of AI definition \nAI has a history much longer than is commonly understood, in fields from science and philosophy ranging all \nthe way back to ancient Greece but its modern iteration owes much to Alan Turing and conference in Dartmouth \nCollege in 1956 where the term “Artificial Intelligence” was officially coined and defined by John McCarthy \nat the time as “the science and engineering of making intelligent machines”. Russel and Norvig (2020) referred \nto it as the “the birth of artificial intelligence.” One of the initial paradigms of AI was that it revolved around \nhigh-level cognition. Not the ability to recognise concepts, perceive objects, or execute complex motor skills \nshared by most animals, but the potential to engage in multi -step reasoning, to understand the meaning of',
  'metadata': {'do

###This is the context  we send to a LLM

Integration VectorDB context pipeline with LLM output

In [33]:
### Simple RAG pipeline with Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Get Groq API key from .env
groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize the Groq LLM
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024
)


# Simple RAG pipeline function
def rag_simple(query, retriever, llm, top_k=3):

    # Retrieve relevant documents
    results = retriever.retrieve(
        query,
        top_k=top_k
    )

    # Combine retrieved documents into one context
    context = "\n\n".join(
        [doc["document"] for doc in results]
    ) if results else ""

    # If nothing relevant was found
    if not context:
        return "No relevant context found to answer the question."

    # Create the prompt
    prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:
"""

    # Send prompt to Groq LLM
    response = llm.invoke(prompt)

    # Return the generated answer
    return response.content

In [34]:
answer = rag_simple(
    "What is Evolution of AI definition?",
    rag_retriever,
    llm
)

print(answer)

Retrieving documents for query: 'What is Evolution of AI definition?'
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.61it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The definition of artificial intelligence has broadened over time.  
- **Early roots** – ideas about intelligent behavior date back to ancient philosophy, but a formal field began with Alan Turing’s work and the 1956 Dartmouth conference.  
- **Original definition (John McCarthy, 1956)** – “the science and engineering of making intelligent machines,” emphasizing high‑level, multi‑step reasoning rather than low‑level perception or motor skills.  
- **Later refinements** – scholars such as Russell & Norvig describe AI as the study of computations that enable **perception, reasoning, and action**, noting that AI need not be limited to biologically‑observable mechanisms.  

Thus, the evolution of the AI definition moves from a narrow focus on symbolic reasoning machines to a broader view that includes any computational approach that gives machines the ability to perceive, reason, and act.


In [35]:
answer = rag_simple(
    "What is  Deep Residual Learning",
    rag_retriever,
    llm
)

print(answer)

Retrieving documents for query: 'What is  Deep Residual Learning'
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.32it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Deep Residual Learning is a neural‑network framework that reformulates each layer (or block of layers) to learn a **residual function** \(F(x)=H(x)-x\) instead of directly learning the desired underlying mapping \(H(x)\). In practice, a shortcut (identity) connection passes the input \(x\) forward and adds it to the output of the stacked layers, so the network actually learns the difference (the “residual”) between the input and the target output.  

This design makes it much easier to train very deep networks (hundreds of layers) because the residual formulation mitigates the degradation problem—deep “plain” networks tend to have higher training error as depth grows—by allowing gradients to flow more directly through the identity shortcuts. The resulting **ResNets** achieve higher accuracy with lower complexity than earlier very deep models such as VGG.


In [39]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['document'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['document'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Explain Bottleneck Architectures?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Explain Bottleneck Architectures?'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.07it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: **Bottleneck architectures** are a variant of the residual block used in deep ResNets to keep computation and memory costs low while allowing very deep networks.

1. **Structure of a bottleneck block**  
   - It consists of **three convolutional layers** instead of the usual two:  
     1. **1 × 1 convolution** – reduces the channel dimension (compression).  
     2. **3 × 3 convolution** – operates on this reduced‑dimensional “bottleneck”.  
     3. **1 × 1 convolution** – restores the original channel dimension (expansion).  
   - The 3 × 3 layer therefore works on a smaller number of channels, saving FLOPs and parameters.

2. **Identity shortcut**  
   - The shortcut connection bypasses the three‑layer stack unchanged (identity).  
   - Because the shortcut links low‑dimensional representations (the reduced‑dimensional space), it adds virtually no extra cost.  
   - If a projection (e.g., 1 × 1 conv) were used instead, the shortcut would have to handle the high‑dimensional e